# Preprocessing check
Quick sanity check of `src/preprocessing.py`: each step on a few examples, then vocab + embedding matrix.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parents[1]))

import nltk
nltk.download("punkt_tab", quiet=True)

from src.data import load_ds, TEXT_COLUMNS
from src.preprocessing import (
    SPECIAL_TOKENS, PAD, UNK, SEP,
    clean_text, tokenize, remove_punctuation, preprocess,
    build_vocab, tokens_to_ids, build_embedding_matrix,
)

ds = load_ds()

## 1. Step by step on hand-picked edge cases

In [ ]:
samples = [
    "She walked into the woods.",
    "He didn’t   know what to do\nnext.",
    "It was Tom´s birthday — \"great!\"",
    "Broken \ufffd char, numbers 3.5 and 100%.",
]
for s in samples:
    c = clean_text(s)
    t = tokenize(c)
    print(repr(s))
    print("  clean   :", repr(c))
    print("  tokens  :", t)
    print("  no punct:", remove_punctuation(t))
    print()

## 2. Real examples from train

In [ ]:
for ex in ds["train"].select(range(3)):
    for col in TEXT_COLUMNS:
        print(f"{col:>14}: {ex[col]}")
        print(f"{'':>14}  {preprocess(ex[col])}")
    print()

## 3. Leftovers after cleaning (should be empty / expected)

In [ ]:
import pandas as pd
from collections import Counter

train_df = ds["train"].to_pandas()
texts = pd.concat([train_df[c] for c in TEXT_COLUMNS], ignore_index=True)
cleaned = texts.map(clean_text)

checks = {
    "non-ascii": r"[^\x00-\x7f]",
    "U+FFFD": "\ufffd",
    "double space": r"\s{2,}",
    "leading/trailing ws": r"^\s|\s$",
}
for name, pat in checks.items():
    hits = cleaned[cleaned.str.contains(pat)]
    print(f"{name:>20}: {len(hits)}", list(hits.head(3)))

non_ascii = Counter(ch for t in cleaned for ch in t if ord(ch) > 127)
non_ascii.most_common(20)

In [ ]:
train_tokens = [preprocess(t) for t in texts]
print("texts with no tokens left:", [t for t, toks in zip(texts, train_tokens) if not toks])
pd.Series([len(t) for t in train_tokens]).describe()

## 4. Vocab + OOV with word2vec

In [ ]:
from src.embeddings import load_word2vec

wv = load_word2vec()
len(wv), wv.vector_size

In [ ]:
vocab = build_vocab(train_tokens, wv)

counts = Counter(t for toks in train_tokens for t in toks)
oov = Counter({t: n for t, n in counts.items() if t not in vocab})
print("vocab size:", len(vocab), "| first entries:", list(vocab)[:8])
print(f"OOV types : {len(oov)} / {len(counts)}")
print(f"OOV tokens: {sum(oov.values()) / sum(counts.values()):.2%}")
oov.most_common(30)

In [ ]:
toks = preprocess(ds["validation"][0]["observation_1"])
print(list(zip(toks, tokens_to_ids(toks, vocab))))

val_df = ds["validation"].to_pandas()
val_tokens = [t for c in TEXT_COLUMNS for text in val_df[c] for t in preprocess(text)]
print(f"validation <unk> rate: {sum(t not in vocab for t in val_tokens) / len(val_tokens):.2%}")

In [ ]:
import numpy as np

E = build_embedding_matrix(vocab, wv)
print("shape:", E.shape, E.dtype)
print("pad all zero:", not E[vocab[PAD]].any(), "| unk all zero:", not E[vocab[UNK]].any())
print("sep norm:", np.linalg.norm(E[vocab[SEP]]).round(3))
w = list(vocab)[len(SPECIAL_TOKENS)]
print(f"row for {w!r} matches wv:", np.allclose(E[vocab[w]], wv[w]))
print("zero rows besides pad/unk:", int((~E.any(axis=1)).sum()) - 2)